<a href="https://colab.research.google.com/github/GabyVothy/Text-Classification/blob/main/1Final_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. BACKGROUND

As we observe the competition between Nike, Adidas, and Lululemon in user's attention on Twitter, this project expects to mapping that competition as a network. Using a dataset of ~150k brand-mention tweets to build a mentions network (who talks to whom, and who's most central or bridges multiple brands) and a semantic network (which words/topics each brand owns vs. shares with competitors), expecting the segment analysis will give us certain insight for marketing activities.

## 2. DATA EXPLORATION

Let's firstly explore data via:
1) install an load necessary tools and library
2) inport dataset

In [1]:
# First let's import necessary
!pip -q install networkx pyvis python-louvain nltk

import json, re, gzip, itertools, os, urllib.request
from collections import Counter, defaultdict

import networkx as nx
import community as community_louvain   # python-louvain
from pyvis.network import Network

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('vader_lexicon', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer

STOP = set(stopwords.words('english'))
LEMM = WordNetLemmatizer()
SIA = SentimentIntensityAnalyzer()

# The three brand Twitter handles as they appear in entities.user_mentions
brand_accounts = {"nike": "Nike", "adidas": "adidas", "lululemon": "lululemon"}
brand_keys = set(brand_accounts.keys())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 89.0 MB/s eta 0:00:00


In [3]:
# Download & load the data
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "drive/MyDrive/Network_Analysis/nikelululemonadidas_tweets.jsonl"

Mounted at /content/drive


In [4]:
# load data
def iter_tweets(path=DATA_PATH):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

def get_text(tw):
    # some tweets are truncated; prefer extended/full_text when present
    if tw.get("truncated") and "extended_tweet" in tw:
        return tw["extended_tweet"].get("full_text", tw.get("text", ""))
    return tw.get("full_text", tw.get("text", "")) or ""

def get_mentions(tw):
    ents = tw.get("entities") or {}
    return ents.get("user_mentions") or []

def tweet_brands(tw):
    found = set()
    for m in get_mentions(tw):
        sn = (m.get("screen_name") or "").lower()
        if sn in brand_keys:
            found.add(sn)
    return found

# quick sanity check
n = 0
for tw in iter_tweets():
    n += 1
    if n >= 200000:
        break
print("Tweets scanned:", n)


Tweets scanned: 175078


In [5]:
# Build a directory of unique users, then we focus on user have at least 5000 followers

unique_users = {}
for tw in iter_tweets():
    u = tw.get("user") or {}
    uid = u.get("id")
    if uid is None:
        continue
    if uid not in unique_users:
        unique_users[uid] = {
            "screen_name": u.get("screen_name"),
            "followers": u.get("followers_count", 0),
            "tweet_count": 0,
            "brands": set(),
        }
    unique_users[uid]["tweet_count"] += 1
    unique_users[uid]["brands"] |= tweet_brands(tw)

print("Total unique users:", len(unique_users))

# narrow the graph on user can have certain influence on their tweets and reduce number of new/clone accounts.
TWEET_MIN = 2 # tweeted about a tracked brand at least twice
FOLLOWER_MIN = 5000 # has a meaningfully sized audience

include_ids = {
    uid for uid, d in unique_users.items()
    if d["tweet_count"] >= TWEET_MIN and d["followers"] >= FOLLOWER_MIN
}
print(f"Users kept after filtering (>= {TWEET_MIN} tweets, >= {FOLLOWER_MIN} followers): {len(include_ids)}"
      f"  ({len(include_ids)/max(1,len(unique_users)):.2%} of all users)")


Total unique users: 104772
Users kept after filtering (>= 2 tweets, >= 5000 followers): 2037  (1.94% of all users)


## 3. BUILD THE TWITTER MENTIONS GRAPH

In [6]:
# Twitter Mentions Graph: ONE directed, valued graph covering all three brands
# Edge: tweeting_user -> mentioned_user/brand, weight = number of times that mention occurred

M = nx.DiGraph()
for tw in iter_tweets():
    u = tw.get("user") or {}
    uid = u.get("id")
    if uid not in include_ids:
        continue
    src = u.get("screen_name")
    for m in get_mentions(tw):
        tgt = m.get("screen_name")
        if not tgt or tgt == src:
            continue
        if M.has_edge(src, tgt):
            M[src][tgt]["weight"] += 1
        else:
            M.add_edge(src, tgt, weight=1)
        M.nodes[src]["node_type"] = "user"
        M.nodes[tgt]["node_type"] = M.nodes[tgt].get("node_type", "user")

    for key, handle in brand_accounts.items():
        if handle in M.nodes:
            M.nodes[handle]["node_type"] = "brand"
            M.nodes[handle]["brand"] = key

print("Mentions graph:", M.number_of_nodes(), "nodes,", M.number_of_edges(), "edges")


Mentions graph: 7267 nodes, 12533 edges


## 4. ANALYSIS FROM GRAPH DATA

Now we have this graph build up to facilitate us on understand the relationship of users, the influencer and their interest on the brands.

Firstly to identify who is most influencer, I will scan for each brand's account node, group the users who mentioned that brand and rank them by the times they mentioned that brand. This approach can help to navigate the user most central relating to the brand not just generally the central of whole graph

### i) central users for marketing campaign

In [7]:
def top_central_users(brand_key, k=10):
    handle = brand_accounts[brand_key]
    if handle not in M:
        return []
    rows = []
    for u in M.predecessors(handle):
        w = M[u][handle]["weight"]
        rows.append((u, w, unique_users.get(_uid_lookup.get(u), {}).get("followers")))
    rows.sort(key=lambda r: r[1], reverse=True)
    return rows[:k]

_uid_lookup = {d["screen_name"]: uid for uid, d in unique_users.items()}

for brand_key in brand_accounts:
    print(f"\n Top 10 most central users for {brand_key} ")
    for screen_name, weight, followers in top_central_users(brand_key, 10):
        print(f"  @{screen_name:<20} mentions={weight:<4} followers={followers}")



 Top 10 most central users for nike 
  @SneakerScouts        mentions=6891 followers=18262
  @JamarroBrown         mentions=129  followers=7864
  @RunBlogRun           mentions=103  followers=35260
  @malachiobrien        mentions=85   followers=41236
  @DaAnsahonSports      mentions=79   followers=50247
  @DavePOfficial        mentions=78   followers=37894
  @x_wx_x               mentions=62   followers=6521
  @Thenizzar            mentions=48   followers=18238
  @HorizonLeague        mentions=47   followers=12726
  @FootwearNews         mentions=39   followers=27747

 Top 10 most central users for adidas 
  @KVSwitzer            mentions=53   followers=9135
  @natmmom              mentions=35   followers=6261
  @FootwearNews         mentions=33   followers=27747
  @JamieGeorge93        mentions=25   followers=5737
  @tw1tte7              mentions=25   followers=11265
  @LEGIQN               mentions=21   followers=318569
  @KicksDeals           mentions=19   followers=467941
  @NFTp

now we have a list of central users on each brand, however there could be bot, or paid users, or real influencer or a superfan. I will explore here a little to utilize also other data relating to user profile, to see if we can identify the user we should focus on this list.

We will combine with also other text in their tweets to group them into their "intention", such as if the Tweets contain majority of Media key words such as blog, press, magazine, ect. then it's likely a media account, and so on.

In [8]:
# keyword/threshold constants
REVIEW_KEYWORDS = ["review", " vs ", "versus", "compared to", "pros and cons",
                    "prefer", "better than", "worse than", "upgrade", "downgrade",
                    "not worth", "out of 10", "/10", "stars"]
MEDIA_KEYWORDS = ["news", "blog", "editor", "reporter", "magazine", "press"]
BOT_KEYWORDS = ["deal", "deals", "alert", "restock", "bot", "drop"]
SENTIMENT_SWING_THRESHOLD = 0.6  #gap between most-positive and most-negative tweet

# extra user-profile fields while building unique_users
unique_users = {}
for tw in iter_tweets():
    u = tw["user"]
    uid = u["id"]
    if uid not in unique_users:
        unique_users[uid] = {
            "screen_name": u.get("screen_name"),
            "followers": u.get("followers_count", 0),
            "friends": u.get("friends_count", 0),
            "statuses_count": u.get("statuses_count", 0),
            "verified": u.get("verified", False),
            "description": (u.get("description") or "").lower(),
            "default_profile_image": u.get("default_profile_image", False),
            "tweet_count": 0,
            "brands": set(),
            "sentiment_scores": [],
            "has_review_keyword": False,
        }
    unique_users[uid]["tweet_count"] += 1
    unique_users[uid]["brands"] |= tweet_brands(tw)
    text = get_text(tw).lower()
    if any(tag in text for tag in ["#ad", "#sponsored", "#gifted", "#partner"]):
        unique_users[uid]["has_disclosure_tag"] = True
    if any(k in text for k in REVIEW_KEYWORDS):
        unique_users[uid]["has_review_keyword"] = True
    unique_users[uid]["sentiment_scores"].append(SIA.polarity_scores(text)["compound"])


def classify_user(d):
    mentions = d["tweet_count"]
    followers = max(d["followers"], 1)
    mention_follower_ratio = mentions / followers
    desc = d["description"]
    scores = d["sentiment_scores"]
    sentiment_swing = (max(scores) - min(scores)) if scores else 0

    if d.get("has_disclosure_tag"):
        return "likely_paid_partnership"
    if any(k in desc for k in BOT_KEYWORDS) or mention_follower_ratio > 0.05:
        return "likely_bot_or_alert_account"
    is_cross_brand = len(d["brands"]) >= 2
    has_mixed_sentiment = sentiment_swing >= SENTIMENT_SWING_THRESHOLD
    if is_cross_brand and (d["has_review_keyword"] or has_mixed_sentiment):
        return "likely_reviewer"
    if d["verified"] or any(k in desc for k in MEDIA_KEYWORDS):
        if is_cross_brand:
            return "likely_media_or_press"
        return "likely_notable_verified_account"
    if d["default_profile_image"] or d["statuses_count"] < 20:
        return "likely_low_effort_or_new_account"
    return "likely_organic_fan"

for uid, d in unique_users.items():
    d["tag"] = classify_user(d)

In [9]:
# update the function to add the likely tendency of the user
def top_central_users(brand_key, k=10):
    handle = brand_accounts[brand_key]
    if handle not in M:
        return []
    rows = []
    for u in M.predecessors(handle):
        w = M[u][handle]["weight"]
        d = unique_users.get(_uid_lookup.get(u), {})
        rows.append((u, w, d.get("followers"), d.get("tag", "unclassified")))
    rows.sort(key=lambda r: r[1], reverse=True)
    return rows[:k]

_uid_lookup = {d["screen_name"]: uid for uid, d in unique_users.items()}

for brand_key in brand_accounts:
    print(f"\n Top 10 most central users for {brand_key} ")
    for screen_name, weight, followers, tag in top_central_users(brand_key, 10):
        print(f"  @{screen_name:<20} mentions={weight:<4} followers={followers:<10} tag={tag}")


 Top 10 most central users for nike 
  @SneakerScouts        mentions=6891 followers=18262      tag=likely_paid_partnership
  @JamarroBrown         mentions=129  followers=7864       tag=likely_reviewer
  @RunBlogRun           mentions=103  followers=35260      tag=likely_reviewer
  @malachiobrien        mentions=85   followers=41236      tag=likely_organic_fan
  @DaAnsahonSports      mentions=79   followers=50247      tag=likely_organic_fan
  @DavePOfficial        mentions=78   followers=37894      tag=likely_paid_partnership
  @x_wx_x               mentions=62   followers=6521       tag=likely_reviewer
  @Thenizzar            mentions=48   followers=18238      tag=likely_paid_partnership
  @HorizonLeague        mentions=47   followers=12726      tag=likely_notable_verified_account
  @FootwearNews         mentions=39   followers=27747      tag=likely_paid_partnership

 Top 10 most central users for adidas 
  @KVSwitzer            mentions=53   followers=9135       tag=likely_paid_par

**so now we have potential intention of these accounts, Marketing team can utilize this information to navigate their approach for proper need, for example discount codes to bots/fans for reach, hint previews to media, review units to reviewers, and partnership conversations to notable/verified accounts — rather than treating every central or bridging account the same way. However, as the graph helps to narrow down the highlighting accounts, it's worth to do some further manual check on them for the tendency before decide concrete strategy as suggested above.**

### ii) Most important bridge for relationship focus

We set a **bridger** as the users/accounts that mention 2+ amongs 3 brands here. Within these bridgers, we rank by **betweenness centrality** in the mentions graph `M` on the users who most often occur on the shortest path between otherwise-separate parts of the network, as the ones structurally connecting brand conversations, not just the ones who happen to tweet a lot.

As we build the tag predicted intention of each account, let's also see what is highlight on the top accounts.

In [ ]:
import pandas as pd

bridge_users = {uid: d for uid, d in unique_users.items()
                if uid in include_ids and len(d["brands"]) >= 2}
print("Number of users mention 2+ amongs 3 brands:", len(bridge_users))

btw = nx.betweenness_centrality(M, weight="weight", normalized=True)

ranked_bridges = sorted(
    bridge_users.items(),
    key=lambda kv: btw.get(kv[1]["screen_name"], 0.0),
    reverse=True,)[:10]

bridge_table = pd.DataFrame([{"screen_name": "@" + d["screen_name"],
                                "brands": ", ".join(sorted(d["brands"])),
                                "betweenness": round(btw.get(d["screen_name"], 0), 6),
                                "followers": d["followers"],
                                "tag": d.get("tag", "unclassified"),
                            } for uid, d in ranked_bridges])

print("\n Top 10 bridgers (by betweenness centrality) ")
bridge_table

Number of users mention 2+ amongs 3 brands: 406


**So the bridger list here is mostly big corporate or media accounts, not individual fans as the name say itself - adidas, BurgerKing, WNBA, BlazePizza show up because they tag or get tagged alongside another brand (likely cross-promotion or sponsorship posts), so marketing should treat these as partnership/co-marketing leads, not outreach targets.
While those tagged as likely_reviewer accounts (i.e boardroom, Adweek, snkr_twitr, nicekicks, DeezeFi, Clarissa_Laskey) are the more useful ones for seeding, the team can consider to send them early product info before launches since they already compare brands in their posts. One thing worth checking by hand: @adidas itself showing up as a bridger between adidas and nike looks odd as it likely means the adidas account tweeted something that also tagged nike, so it's worth a quick look to see if that's a real signal or just noise in how the data was built.**

### iii) Sentiment Segmentation

We score every tweet's sentiment with VADER, then rebuild the mentions network
separately for **positive** and **negative** tweets to see if different users/structures
dominate praise vs. complaints.

In [ ]:
import pandas as pd
from IPython.display import display

def sentiment_label(text, pos_thresh=0.3, neg_thresh=-0.3):
    score = SIA.polarity_scores(text)["compound"]
    if score >= pos_thresh:
        return "positive"
    if score <= neg_thresh:
        return "negative"
    return "neutral"

def build_mentions_graph(tweet_filter):
    g = nx.DiGraph()
    for tw in iter_tweets():
        u = tw.get("user") or {}
        uid = u.get("id")
        if uid not in include_ids:
            continue
        if not tweet_filter(tw):
            continue
        src = u.get("screen_name")
        for m in get_mentions(tw):
            tgt = m.get("screen_name")
            if not tgt or tgt == src:
                continue
            if g.has_edge(src, tgt):
                g[src][tgt]["weight"] += 1
            else:
                g.add_edge(src, tgt, weight=1)
    return g

POS_CACHE, NEG_CACHE = {}, {}

def is_positive(tw):
    return sentiment_label(get_text(tw)) == "positive"

def is_negative(tw):
    return sentiment_label(get_text(tw)) == "negative"

M_pos = build_mentions_graph(is_positive)
M_neg = build_mentions_graph(is_negative)

network_summary = pd.DataFrame([
    {"network": "positive", "nodes": M_pos.number_of_nodes(), "edges": M_pos.number_of_edges()},
    {"network": "negative", "nodes": M_neg.number_of_nodes(), "edges": M_neg.number_of_edges()},
])


brand_mentions_rows = []
for name, g in [("positive", M_pos), ("negative", M_neg)]:
    for key, handle in brand_accounts.items():
        if handle in g:
            brand_mentions_rows.append({
                "network": name,
                "brand": handle,
                "weighted_in_degree": g.in_degree(handle, weight="weight"),
            })

brand_mentions_table = pd.DataFrame(brand_mentions_rows).pivot(
    index="brand", columns="network", values="weighted_in_degree"
).fillna(0).astype(int)

# retrieve result
print("Network sizes:")
display(network_summary)

print("\nMost-mentioned brand accounts, positive vs negative:")
display(brand_mentions_table)

### iv) Influencer Segmentation by follower count

Restrict the mentions graph to tweets that *originate* from high-follower accounts only,
so we can see how the "elite"/influential conversation differs from the general one.

In [ ]:
INFLUENCER_FOLLOWER_MIN = 100000
influencer_ids = {uid for uid, d in unique_users.items() if d["followers"] >= INFLUENCER_FOLLOWER_MIN}
print("Influencer accounts (>= {:,} followers):".format(INFLUENCER_FOLLOWER_MIN), len(influencer_ids))

def is_from_influencer(tw):
    return (tw.get("user") or {}).get("id") in influencer_ids

M_influencers = build_mentions_graph(is_from_influencer)
print("Influencer-only network:", M_influencers.number_of_nodes(), "nodes,", M_influencers.number_of_edges(), "edges")


### v) PyVis visualization on Mentions network

Now we build the Mentions network in PyVis to visuallize the connection/relation and centralization per 1) all brands, 2) positive, 3) negative, 4) influencer.
In which design will follow:
- Node color = brand the account belongs to / mostly mentions;
- Node size = weighted on how much they're talked about;
- Brand accounts are highlighted.


In [ ]:
BRAND_COLORS = {"nike": "#F26522", "adidas": "#000000", "lululemon": "#E6007E", None: "#9AA5B1"}

def node_brand(node):
    if node in brand_accounts.values():
        for k, v in brand_accounts.items():
            if v == node:
                return k
    d = unique_users.get(_uid_lookup.get(node))
    if d and d["brands"]:
        counts = Counter(d["brands"])
        return counts.most_common(1)[0][0]
    return None

def draw_mentions_pyvis(graph, filename="mentions_network.html", max_nodes=400):
    g = graph
    if g.number_of_nodes() > max_nodes:
        deg = dict(g.degree(weight="weight"))
        keep = set(sorted(deg, key=deg.get, reverse=True)[:max_nodes])
        g = g.subgraph(keep).copy()

    net = Network(height="800px", width="100%", directed=True, notebook=True, cdn_resources="in_line")
    for node in g.nodes():
        brand = node_brand(node)
        size = 15 + 2 * g.degree(node, weight="weight")
        is_brand_acct = node in brand_accounts.values()
        net.add_node(
            node, label=node,
            color=BRAND_COLORS.get(brand, BRAND_COLORS[None]),
            size=size * (3 if is_brand_acct else 1),
            title=f"{node} | brand={brand} | weighted degree={g.degree(node, weight='weight')}",
        )
    for u, v, data in g.edges(data=True):
        net.add_edge(u, v, value=data.get("weight", 1))

    net.show_buttons(filter_=["physics"])
    display(net.show(filename, notebook=True))

draw_mentions_pyvis(M, "mentions_network_all_brands.html")
draw_mentions_pyvis(M_influencers, "mentions_network_influencers.html")

**now we have more dynamic playgound for marketing team to analyze and dive into more detail as needed — not just the obvious top 10, but also smaller, positive accounts that the ranked list alone would miss. This matters because the top-10 tables only surface the loudest accounts (often bots, media, or mega-influencers), while the graph lets you spot smaller likely_organic_fan or likely_reviewer nodes sitting close to a brand hub — accounts worth a gift/sample seeding campaign rather than a paid-per-click spend, since the goal there is reaching a specific, genuine community, not buying mass reach from a major influencer or media account.**

## 5. BUILD SEMANTIC NETWORK GRAPH


**second part of this project we focus on topic that people in the tweets discussed about, using topic model to identify the topics links to brands in our observation**

### i) Build the combined semantic graph
we will build 1 graph for all 3 brandse and keep word-frequency counters on each brand so we can later determine which words each brand "owns" vs. shares with competitors.

In [ ]:
import re
import string
import itertools
import networkx as nx
from collections import Counter, defaultdict
from nltk import TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

LEMM = WordNetLemmatizer()

TWEET_TOKENIZER = TweetTokenizer().tokenize
STOPWORDS = set(stopwords.words("english")) # added additional words
URL_FRAGMENTS = {"htt", "http", "https", "www", "co", "ps", "tco"}
EXTRA_STOP = {"amp", "rt","via","ad"} | {h.lower() for h in brand_accounts.values()}

def is_punct_token(t):
    return all(ch in string.punctuation for ch in t)

def clean_tokens(text):
    text = re.sub(r"https?://\S*", " ", text)
    tokens = TWEET_TOKENIZER(text.lower())
    tokens = [t for t in tokens if "http" not in t and "://" not in t and t not in URL_FRAGMENTS]
    tokens = [t for t in tokens if not is_punct_token(t)]
    tokens = [t for t in tokens if t not in STOPWORDS]        # <-- this line was missing
    tokens = [t.lstrip("@").lstrip("#") for t in tokens]
    tokens = [t for t in tokens if t not in STOP and t not in EXTRA_STOP]
    tokens = [t for t in tokens if len(t) >= 2 and not t.isdigit()]
    tokens = [LEMM.lemmatize(t) for t in tokens]
    return tokens

word_counts = Counter()
brand_word_counts = defaultdict(Counter)
edge_weights = Counter()

for tw in iter_tweets():
    u = tw.get("user") or {}
    if u.get("id") not in include_ids:
        continue
    toks = clean_tokens(get_text(tw))
    if not toks:
        continue
    uniq_toks = sorted(set(toks))
    word_counts.update(uniq_toks)
    brands = tweet_brands(tw)
    for b in brands:
        brand_word_counts[b].update(uniq_toks)
    for a, b_ in itertools.combinations(uniq_toks, 2):
        edge_weights[(a, b_)] += 1

print("Unique words after preprocessing:", len(word_counts))

# frequency floor so the network stays interpretable which we aim to around 500 nodes
MIN_WORD_FREQ = 30   # at least 50 words counted
keep_words = {w for w, c in word_counts.items() if c >= MIN_WORD_FREQ}
print(f"Words kept at MIN_WORD_FREQ={MIN_WORD_FREQ}:", len(keep_words))

graph = nx.Graph()
for (a, b_), w in edge_weights.items():
    if a in keep_words and b_ in keep_words:
        graph.add_edge(a, b_, weight=w)
print("Semantic graph:", graph.number_of_nodes(), "nodes,", graph.number_of_edges(), "edges")


### 2. Topic detection by semantic word clusters

we will use community_louvain with Louvain algo to find the community in a network build above. The better we clean the text, the more coherent we got from community topic here

In [ ]:
LOUVAIN_RESOLUTION = 1.3
partition = community_louvain.best_partition(graph, weight="weight", resolution=LOUVAIN_RESOLUTION, random_state=42)
n_communities = len(set(partition.values()))
print("Number of word communities found:", n_communities)

communities = defaultdict(list)
for word, cid in partition.items():
    communities[cid].append(word)

def top_words_in_community(cid, k=10):
    return sorted(communities[cid], key=lambda w: graph.degree(w, weight="weight"), reverse=True)[:k]

community_table = pd.DataFrame([
    {"community": cid, "size": len(communities[cid]), "top_words": ", ".join(top_words_in_community(cid))}
    for cid in sorted(communities, key=lambda c: -len(communities[c]))[:n_communities]
])

with pd.option_context("display.max_colwidth", None):
    display(community_table)

**Looking at these word clusters, we can see that our preprocessing is working just okay, contain certain noise in the topic. The biggest cluster is general sneaker and basketball talk, which is the core, everyday conversation around the brand. Next to it, we see a retail cluster — words like store names and shoe models — this is literally product-drop and "where to buy" conversation. Then later clusters contains new release and collab hype, which is a different kind of excitement from just restock info or running and marathons, a totally different audience from the sneakerhead crowd, so they'd need a different marketing or even PR activitapproach**

It can be done better to give more coherent info.

### 3. What the brands own

For every kept word, compute its share of mentions across the three brands.
A word is "owned" by a brand when the overwhelming majority of its usage occurs
in tweets about that brand and it appears rarely for the others.

In [ ]:
def brand_ownership(word):
    counts = {b: brand_word_counts[b][word] for b in brand_accounts}
    total = sum(counts.values())
    if total == 0:
        return None
    top_brand = max(counts, key=counts.get)
    share = counts[top_brand] / total
    return top_brand, share, total, counts

brand_total_words = {b: sum(brand_word_counts[b].values()) for b in brand_accounts}
OWNERSHIP_SHARE_THRESHOLD = 0.6
MIN_TOTAL_MENTIONS = 50

owned = defaultdict(list)
for w in keep_words:
    res = brand_ownership(w)
    if res is None:
        continue
    top_brand, share, total, counts = res
    if total >= MIN_TOTAL_MENTIONS and share >= OWNERSHIP_SHARE_THRESHOLD:
        rate_per_1k = counts[top_brand] / brand_total_words[top_brand] * 1000
        owned[top_brand].append((w, share, total, rate_per_1k))

owned_table = pd.DataFrame([
    {"brand": brand, "word": w, "share": f"{share:.0%}", "total_mentions": total, "rate_per_1k_words": round(rate, 2)}
    for brand in brand_accounts
    for w, share, total, rate in sorted(owned[brand], key=lambda r: r[3], reverse=True)[:5]
])
owned_table

Nike owns retail/availability conversation at high volume, while adidas owns smaller but more campaign-distinctive conversation

### 4. How the bridging words saying?

Like bridger users, bridging words show up across 2+ brands — meaning consumers already link that word to the whole category, not one brand. So brands shouldn't build ads around these shared words; the real differentiation comes from the brand-owned words instead.

In [ ]:
def bridging_score(word):
    counts = {b: brand_word_counts[b][word] for b in brand_accounts}
    total = sum(counts.values())
    if total < MIN_TOTAL_MENTIONS:
        return None
    nonzero = [c for c in counts.values() if c > 0]
    if len(nonzero) < 2:
        return None
    # balance = how evenly split across brands (1.0 = perfectly even across brands present)
    max_share = max(counts.values()) / total
    balance = 1 - max_share
    return balance, total, counts

bridging = []
for w in keep_words:
    res = bridging_score(w)
    if res:
        balance, total, counts = res
        bridging.append((w, balance, total, counts))

bridging.sort(key=lambda r: (r[1], r[2]), reverse=True)

bridging_table = pd.DataFrame([
    {"word": w, "balance": round(balance, 2), "total_mentions": total, **counts}
    for w, balance, total, counts in bridging[:10]
])
bridging_table

### 5. PyVis Visualization for Semantic Network

Now we use PyVis again to visuallize the graph on this topic/semantic network

In [ ]:
import colorsys

def community_color(cid, total):
    hue = (cid % max(total, 1)) / max(total, 1)
    r, g, b = colorsys.hsv_to_rgb(hue, 0.65, 0.85)
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

def draw_semantic_pyvis(graph, partition, filename="semantic_network.html", max_nodes=500, height="700px"):
    g = graph
    if g.number_of_nodes() > max_nodes:
        deg = dict(g.degree(weight="weight"))
        keep = set(sorted(deg, key=deg.get, reverse=True)[:max_nodes])
        g = g.subgraph(keep).copy()

    total_comms = len(set(partition.values())) or 1
    net = Network(height=height, width="100%", directed=False, notebook=True, cdn_resources="in_line")
    for node in g.nodes():
        cid = partition.get(node, 0)
        net.add_node(
            node, label=node,
            color=community_color(cid, total_comms),
            size=10 + 1.5 * g.degree(node, weight="weight"),
            title=f"{node} | community={cid} | weighted degree={g.degree(node, weight='weight')}",
        )
    for u, v, data in g.edges(data=True):
        net.add_edge(u, v, value=data.get("weight", 1))

    net.toggle_physics(False)
    net.show_buttons(filter_=["physics"])
    display(net.show(filename, notebook=True))

draw_semantic_pyvis(graph, partition, "semantic_network_all_brands.html")

def draw_single_community(cid, filename=None):
    words = communities[cid]
    sub = graph.subgraph(words).copy()
    draw_semantic_pyvis(sub, partition, filename or f"semantic_community_{cid}.html", height="500px")

## 6. CONCLUSION

Network can be built on different elements or data specialty, such as user/relationship with brand or background/topic exploration. From such network we can have more insightful mapping across the dataset, which give more benefit to business for action plan.

**Insight**
- Mentions network: Nike/adidas dominate volume, lululemon is thin; top accounts are a mix of bots, media, and real fans — each needs different outreach, not one blanket approach
- Semantic network: Nike leads on generic retail chatter, adidas owns more distinctive campaign language; shared words (gift, shop, collection) aren't worth differentiating ads around
- Watch-outs: a recurring China/labor-controversy cluster (real signal, flag to PR) and a username-only cluster (possible bot ring, needs manual check)

**Recommended Actions**
- Nike should consider to redirect some paid-partnership spend (e.g. SneakerScouts) into reviewer-seeding — same low-cost trust play adidas/lululemon get for free; also build on retail volume with more campaign-specific messaging, not just availability chatter
- Suggest: don't build differentiated ads around these most common words detected in Bridge words but rather invest in brand-owned words instead

**Future works**
- Tune the semantic model further
- Look closer at each network for more insightful decision.
